In [117]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import os 
import sys

In [20]:
root_dir = os.path.dirname(os.getcwd())

order_path = os.path.join(root_dir, "data", "fct_orders.csv")
orders = pd.read_csv(order_path)

customer_path = os.path.join(root_dir, "data", "dim_customers.csv")
customers = pd.read_csv(customer_path)

product_path = os.path.join(root_dir, "data", "dim_products.csv")
products = pd.read_csv(product_path)

# 1. Overal exploration

### 1.1 Check overview

In [45]:
def overview_check(df):
    print(f"Number of rows: {len(df)}")
    print(f"Number of columns: {len(df.columns.tolist())}")
    print(f"Column list: {df.columns.tolist()}")
    print("\nSample rows:")

    return df.head()

In [56]:
overview_check(orders)

Number of rows: 10000
Number of columns: 8
Column list: ['brand', 'item_id', 'gross_item_value', 'order_date', 'order_datetime', 'product_id', 'order_id', 'customer_id']

Sample rows:


,brand,item_id,gross_item_value,order_date,order_datetime,product_id,order_id,customer_id
0,Brand 1,1370486,21.50,1/1/2024,2024-01-01T10:29:14,2077,1229709,5.800000e+18
1,Brand 2,3794133,4.38,1/1/2024,2024-01-01T23:30:21,119,753517,8.210000e+18
2,Brand 3,17037113,40.50,1/1/2024,2024-01-01T13:42:54.547000,5145,2994406,1.150000e+18
3,Brand 4,17037214,45.91,1/1/2024,2024-01-01T14:50:30.140000,17013,2994429,2.960000e+18
4,Brand 2,3792949,5.05,1/1/2024,2024-01-01T06:56:35,1095,753292,6.720000e+18


In [48]:
overview_check(customers)

Number of rows: 7847
Number of columns: 3
Column list: ['customer_id', 'country', 'state']

Sample rows:


,customer_id,country,state
0,1.270000e+18,AE,Other
1,6.690000e+18,AE,Other
2,2.320000e+18,AG,Other
3,5.340000e+18,AT,Other
4,7.700000e+18,AU,Other


In [49]:
overview_check(products)

Number of rows: 3115
Number of columns: 2
Column list: ['product_id', 'product_group']

Sample rows:


,product_id,product_group
0,1407,Other
1,1673,Other
2,1687,Other
3,1709,Other
4,1743,Other


**Interpretation**
- Column names are consistent across the tables,  no need to modify
- orders can be merged with customers on customer_id, and can be merged with products on product_id. However, before merging, we need to check data quality

### 1.2 Column type

In [50]:
orders.dtypes.to_frame("dtype")

,dtype
brand,str
item_id,int64
gross_item_value,float64
order_date,str
order_datetime,str
product_id,int64
order_id,int64
customer_id,float64


In [51]:
customers.dtypes.to_frame("dtype")

,dtype
customer_id,float64
country,str
state,str


In [52]:
products.dtypes.to_frame("dtype")

,dtype
product_id,int64
product_group,str


**Interpretation**
- Every the data type matches expectation well, no need to modify

### 1.3 Descriptive statistics

In [55]:
orders['gross_item_value'].describe()

count    10000.000000
mean        30.661318
std        200.365242
min         -2.860000
25%          4.787500
50%         12.480000
75%         24.122500
max      15064.350000
Name: gross_item_value, dtype: float64

# 2. Check data quality & clean
* This stage is needed before merging datasets to generate the master data

### 1.1 Missing value check

In [108]:
orders.isnull().sum()

brand               0
item_id             0
gross_item_value    0
order_date          0
order_datetime      0
product_id          0
order_id            0
customer_id         0
dtype: int64

In [109]:
products.isnull().sum()

product_id       0
product_group    0
dtype: int64

In [110]:
customers.isnull().sum()

customer_id    0
country        0
state          0
dtype: int64

**Interpretation**
- There is no missing values in orders, products and customers

### 1.2 Duplicate check

In [92]:
def check_duplication(df, primary_key):
    # Full-row duplicates
    n_full_dups = df.duplicated().sum()
    print(f"Full-row duplicates   : {n_full_dups} : {n_full_dups/len(df):.2f} %")

    # Duplicate primary key
    if primary_key: 
        n_id_dups = df.duplicated(subset=[primary_key]).sum()
        print(f"Duplicate {primary_key}: {n_id_dups}")

In [93]:
check_duplication(orders, 'item_id')

Full-row duplicates   : 0 : 0.00 %
Duplicate item_id: 0


In [94]:
check_duplication(products, 'product_id')

Full-row duplicates   : 0 : 0.00 %
Duplicate product_id: 0


In [95]:
check_duplication(customers, None)

Full-row duplicates   : 925 : 0.12 %


**Interpretation**
- There is no duplication in orders and products
- There are 925 duplicate rows (12% data size) in customers, again, this table has no primary key

In [98]:
def drop_duplication(df, id_col=None, sorted_col=None):
    n_before = len(df)
    # For full-row duplicates
    df = df.drop_duplicates()

    # For IDs duplicate, keep the most recent record
    if id_col and sorted_col:
        df = df.sort_values(sorted_col, ascending=False).drop_duplicates(
            subset=[id_col], keep="first"
        )
    print(
        f"\nRows before: {n_before:,}  →  after dedup: {len(df):,}  (removed {n_before - len(df):,})"
    )

In [99]:
drop_duplication(customers)


Rows before: 7,847  →  after dedup: 6,922  (removed 925)


### 1.3 Cardinallity check

In [57]:
def cardinality_check(df, columns):
    results = []

    for col in columns:
        if col in df.columns:
            results.append(
                {
                    "column": col,
                    "unique_values": df[col].nunique(dropna=False),
                }
            )

    result_df = pd.DataFrame(results).sort_values(by="unique_values", ascending=False)

    print("\nCardinality Check Table:\n")

    return result_df

In [66]:
print(f"Number of rows in orders.csv: {len(orders)}")
cardinality_check(orders, orders.columns)

Number of rows in orders.csv: 10000

Cardinality Check Table:



,column,unique_values
1,item_id,10000
6,order_id,8744
4,order_datetime,8742
2,gross_item_value,4170
5,product_id,3115
7,customer_id,1450
3,order_date,60
0,brand,7


In [65]:
print(f"Number of rows in products.csv: {len(products)}")
cardinality_check(products, products.columns)

Number of rows in products.csv: 3115

Cardinality Check Table:



,column,unique_values
0,product_id,3115
1,product_group,11


In [67]:
print(f"Number of rows in customers.csv: {len(customers)}")
cardinality_check(customers, customers.columns)

Number of rows in customers.csv: 7847

Cardinality Check Table:



,column,unique_values
0,customer_id,1450
2,state,62
1,country,37


**Interpretation**
- item_id (but not order_id) is the primary key of orders.csv
- product_id is the primary key of products.csv
- There is no primary key in customers.csv, customer_id has count being lower than length of data

### 1.4 Outlier check

In [115]:
def outlier_analysis(df, numeric_cols):
    outlier_summary = []
    for col in numeric_cols:
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = ((s < lo) | (s > hi)).sum()
        z = np.abs(stats.zscore(s))
        n_z = (z > 3).sum()
        outlier_summary.append(
            {
                "column": col,
                "mean": s.mean(),
                "median": s.median(),
                "std": s.std(),
                "min": s.min(),
                "max": s.max(),
                "IQR_low": lo,
                "IQR_high": hi,
                "n_IQR_outliers": n_out,
                "pct_IQR_outliers": round(n_out / len(s) * 100, 2),
                "n_zscore>3": n_z,
            }
        )

    outlier_df = pd.DataFrame(outlier_summary)
    print("=== Outlier Summary ===")
    return outlier_df

In [118]:
outlier_analysis(orders, 'gross_item_value')

KeyError: 'g'